In [1]:
import zipfile, json
from pathlib import Path
I = Path("/kaggle/input")
want = {"ckpt_test": "qids.npy", "prep_test": "q.parquet", "ckpt_b_test": "pairs_b.npz",
        "ckpt_b_train": "pairs_b.npz", "ckpt_c_test": "pairs_c.npz"}
seen = {}
for f in I.rglob("*"):                                  # unpacked by Kaggle
    for name, marker in want.items():
        if f.name == marker and f.parent.parent.name == name:
            seen.setdefault(name, set()).add(f.parent.name)
for z in I.rglob("*.zip"):                              # kept as zips
    for n in zipfile.ZipFile(z).namelist():
        parts = n.split("/")
        for name, marker in want.items():
            if name in parts and parts[-1] == marker:
                seen.setdefault(name, set()).add(parts[parts.index(name) + 1])
for name in want:
    print(f"{name:<13} {sorted(seen.get(name, [])) or '-'}")
print("translations_test.parquet:", [str(p) for p in I.rglob("translations_test.parquet")] or "NOT FOUND")
print("competition dataset:", "found" if list(I.rglob("test_source1.tsv")) else "NOT FOUND")

ckpt_test     ['France', 'India', 'US']
prep_test     ['France', 'India', 'US']
ckpt_b_test   ['France', 'India', 'US']
ckpt_b_train  -
ckpt_c_test   ['France', 'India', 'US']
translations_test.parquet: ['/kaggle/input/datasets/eshabachhav/phasec-blocking-predict-input/translations_test.parquet']
competition dataset: found


In [2]:
# CELL 0 (new notebook only): place ckpt_test/ and prep_test/ in /kaggle/working,
# where Step 2 v3 reads them. Works whether Kaggle unpacked the uploaded zips or not.
import shutil, zipfile
from pathlib import Path

W, I = Path("/kaggle/working"), Path("/kaggle/input")

def place(name, marker):
    dest = W / name
    found = {}
    for f in sorted(I.rglob(marker)):                     # unpacked uploads
        if f.parent.parent.name == name:
            found.setdefault(f.parent.name, f.parent)
    for z in sorted(I.rglob("*.zip")):                   # uploads kept as zips
        with zipfile.ZipFile(z) as zf:
            members = [n for n in zf.namelist() if n.startswith(f"{name}/")]
            countries = {n.split("/")[1] for n in members if n.count("/") >= 2}
            if members and not countries <= set(found):
                zf.extractall(W)
                for c in countries:
                    found.setdefault(c, W / name / c)
    for c, src in sorted(found.items()):
        if not (dest / c).exists():
            shutil.copytree(src, dest / c)
    if not dest.exists():
        print(f"{name}: NOT FOUND — attach its dataset"); return
    for c in sorted(p.name for p in dest.iterdir() if p.is_dir()):
        files = sorted(f.name for f in (dest / c).iterdir())
        size = sum(f.stat().st_size for f in (dest / c).iterdir()) / 1e9
        print(f"{name}/{c:<7} {size:5.2f} GB   {', '.join(files)}")

place("ckpt_test", "qids.npy")
place("prep_test", "q.parquet")
used = sum(f.stat().st_size for f in W.rglob("*") if f.is_file()) / 1e9
print(f"/kaggle/working: {used:.1f} GB used of 20 GB")

ckpt_test/France   0.59 GB   pairs.npz, pids.npy, qids.npy, stats.json
ckpt_test/India    1.92 GB   pairs.npz, pids.npy, qids.npy, stats.json
ckpt_test/US       1.45 GB   pairs.npz, pids.npy, qids.npy, stats.json
prep_test/France   0.17 GB   p.parquet, p_x3.parquet, q.parquet, q_x3.parquet
prep_test/India    0.73 GB   p.parquet, p_x3.parquet, q.parquet, q_x3.parquet
prep_test/US       0.58 GB   p.parquet, p_x3.parquet, q.parquet, q_x3.parquet
/kaggle/working: 5.6 GB used of 20 GB


In [3]:
SMOKE = False
SMOKE_QUERIES = 20_000
DEFAULT_VARIANT = "auto"      # "auto" = base_owner if Step 1 v3 found the rule helps, else base
 
DATA_ROOT = "/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset"
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
 
Q_CHUNK = 50_000
PRED_CHUNK = 5_000_000
FEAT_CHUNK = 2_000_000
CALIB_GRID = [0.001, 0.002, 0.005] + [round(x / 100, 2) for x in range(1, 100)]

In [4]:
# %% [CELL 2] Imports and helpers
# ---------------------------------------------------------------------------
# Logging, timing, zip-aware input finder.
# ---------------------------------------------------------------------------
import gc
import hashlib
import json
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
 
import numpy as np
import pandas as pd
import psutil
import lightgbm as lgb
from tqdm.auto import tqdm
 
try:
    from rapidfuzz.process import cpdist
except ImportError:
    print("installing/upgrading rapidfuzz (needs Internet ON) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "rapidfuzz"], check=True)
    try:
        from rapidfuzz.process import cpdist
    except ImportError:
        raise SystemExit("rapidfuzz was upgraded: restart the session, then rerun this cell.")
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
 
DATA_ROOT, WORK, INPUT_ROOT = Path(DATA_ROOT), Path(WORK), Path(INPUT_ROOT)
T_START = time.time()
TIMES = {}                     # stage -> [seconds, scalable?]
 
 
def log(msg):
    print(f"  [{(time.time()-T_START)/60:5.1f} min] {msg}", flush=True)
 
 
def mem(tag=""):
    rss = psutil.Process().memory_info().rss / 1e9
    print(f"    [MEM] {tag:<28} RAM={rss:5.1f} GB  free={psutil.virtual_memory().available/1e9:5.1f} GB",
          flush=True)
 
 
def bar(it=None, **kw):
    kw.setdefault("mininterval", 10)
    return tqdm(it, **kw)
 
 
def timed(stage, t0, scalable=True):
    """Accumulate wall time per stage. scalable: True = grows with the number of
    pairs, False = fixed cost, None = done once and reused by the full run."""
    TIMES.setdefault(stage, [0.0, scalable])[0] += time.time() - t0
 
 
def hash_bucket(eid):
    """Deterministic 0-9999 bucket; identical to the EDA / Notebook 2 split."""
    return int(hashlib.md5(eid.encode()).hexdigest(), 16) % 10000
 
 
def locate(marker):
    """All files named `marker` under INPUT_ROOT, plus those inside any .zip under
    INPUT_ROOT. Each zip holding the marker is unpacked once into
    WORK/_unzipped/<zip name>/ (Kaggle may keep uploaded zips packed)."""
    root_ok = INPUT_ROOT.exists()
    hits = sorted(INPUT_ROOT.rglob(marker)) if root_ok else []
    for z in (sorted(INPUT_ROOT.rglob("*.zip")) if root_ok else []):
        dest = WORK / "_unzipped" / z.stem
        done = dest / ".unpacked"
        if not done.exists():
            with zipfile.ZipFile(z) as zf:
                if not any(n.endswith("/" + marker) or n == marker for n in zf.namelist()):
                    continue
                log(f"unpacking {z.name} -> {dest}")
                zf.extractall(dest)
            done.touch()
        hits += sorted(dest.rglob(marker))
    return hits
 
 
def project(full_factor, label):
    """Print measured stage times and their projection to the full run."""
    print(f"\n  RUNTIME PROJECTION ({label})")
    print(f"    {'stage':<30}{'smoke (s)':>11}{'full (min)':>12}")
    total = 0.0
    for stage, (sec, scalable) in TIMES.items():
        # scalable None = already done and reused by the full run
        full = 0.0 if scalable is None else sec * (full_factor if scalable else 1.0)
        total += full
        print(f"    {stage:<30}{sec:>11.1f}{full/60:>12.1f}")
    print(f"    {'TOTAL':<30}{'':>11}{total/60:>12.1f}")
    return total

In [5]:
# %% [CELL 3] Text cleaning (shared with every stage)
# ---------------------------------------------------------------------------
# Identical cleaning everywhere.
# ---------------------------------------------------------------------------
COLS = ["entity_id", "business_name", "business_address", "country"]
 
 
def load_src(path):
    """Read a source TSV. sep='\t' is mandatory: fields contain commas."""
    return pd.read_csv(path, sep="\t", dtype=str, names=COLS, header=0, keep_default_na=False)
 
 
# ---- cleaning: copied verbatim from Notebook 2 so both stages agree ----------
LEGAL_RE = (r"\b(llc|l\.l\.c|inc|corp|corporation|ltd|limited|pvt|private|"
            r"llp|lp|pllc|sas|sarl|eurl|sasu|snc|company)\b")
 
ADDR_ABBR = [
    (r"\brd\b", "road"), (r"\bst\b", "street"), (r"\bdr\b", "drive"),
    (r"\bave\b", "avenue"), (r"\bblvd\b", "boulevard"), (r"\bln\b", "lane"),
    (r"\bct\b", "court"), (r"\bcir\b", "circle"), (r"\bhno\b", "house number"),
    (r"\bno\b", "number"), (r"\bopp\b", "opposite"), (r"\bnr\b", "near"),
    (r"\bdist\b", "district"), (r"\bflr\b", "floor"),
]
 
 
def clean_text(s):
    """Shared cleaning. Non-Latin scripts are dropped here; L3 reads raw text."""
    s = s.fillna("").astype(str).str.lower()
    s = (s.str.normalize("NFKD")
          .str.encode("ascii", errors="ignore")
          .str.decode("ascii"))
    s = s.str.replace(r"##\s*", "", regex=True)
    s = s.str.replace(r"\bwww\.", "", regex=True)
    s = s.str.replace(r"\.(com|net|org|co|in|fr|uk|biz|info|io)\b", "", regex=True)
    s = s.str.replace(r"^[\s\-\.\,\!\@\#\$\%\^\&\*\>\<\'\"/\\]+", "", regex=True)
    s = s.str.replace("&", " and ", regex=False)
    s = s.str.replace(r"\bn/?a\b", " ", regex=True)
    s = s.str.replace(r"[^\w\s]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()
 
 
def clean_name(s):
    """Name: shared cleaning + legal suffixes + repeated words (FH FH -> FH)."""
    s = clean_text(s)
    s = s.str.replace(LEGAL_RE, " ", regex=True)
    s = s.str.replace(r"\b(\w+)\s+\1\b", r"\1", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()
 
 
def clean_addr(s):
    """Address: shared cleaning + abbreviation expansion."""
    s = clean_text(s)
    for pat, rep in ADDR_ABBR:
        s = s.str.replace(pat, rep, regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()
 
 
# ---- extra text columns for Stage 2 -------------------------------------------
# Street-type words and French articles are skipped when forming the address key,
# so "15 Rue de la Hoxie" -> "15_hoxie" (not "15_rue"), matching "351 Hoxie Avenue"
# style keys. This matters most for France, which training never sees.
SKIP_AFTER_NUMBER = {
    "road", "street", "drive", "avenue", "boulevard", "lane", "court", "circle",
    "way", "highway", "rue", "impasse", "allee", "chemin", "place", "route", "quai",
    "cours", "square", "de", "du", "des", "la", "le", "les", "l", "d",
}
 
 
def addr_key_string(caddr):
    """House number + the first following token that isn't a street word/article.
    Returned as a space-joined, sorted string so rapidfuzz can compare two of them."""
    toks = caddr.split()
    keys = set()
    for i, a in enumerate(toks):
        if a.isdigit():
            a2 = a.lstrip("0")
            j = i + 1
            while j < len(toks) and toks[j] in SKIP_AFTER_NUMBER:
                j += 1
            if a2 and j < len(toks):
                b = toks[j].lstrip("0") if toks[j].isdigit() else toks[j]
                if b:
                    keys.add(a2 + "_" + b)
    return " ".join(sorted(keys))
 
 
def prepare_texts(df):
    """All per-record text columns Stage 2 needs, computed once."""
    out = pd.DataFrame({"entity_id": df["entity_id"].values})
    out["cname"] = clean_name(df["business_name"]).values
    out["caddr"] = clean_addr(df["business_address"]).values
    out["digits"] = out["caddr"].str.findall(r"\b\d+\b").str.join(" ")
    out["nospace"] = out["cname"].str.replace(" ", "", regex=False)
    out["akeys"] = [addr_key_string(a) for a in out["caddr"]]
    out["nonlatin"] = np.array([any(ord(ch) > 0x024F for ch in s)
                                for s in df["business_name"]], dtype=np.int8)
    return out

In [6]:
# %% [CELL 4] String features (shared with Steps 1/2)
# ---------------------------------------------------------------------------
# rapidfuzz similarities + flags.
# ---------------------------------------------------------------------------
# ---- feature definitions (shared with Step 2) --------------------------------
def group_features(q, S):
    """Scores relative to each S1's own candidate list (q must be sorted).
    Scale-free features like rank and gap-to-best transfer across countries."""
    starts = np.flatnonzero(np.r_[True, q[1:] != q[:-1]])
    sizes = np.diff(np.r_[starts, len(q)])
    cols = []
    for j in range(S.shape[1]):
        s = S[:, j].astype(np.float32)
        gmax = np.repeat(np.maximum.reduceat(s, starts), sizes)
        order = np.lexsort((-s, q))
        rank = np.empty(len(q), np.float32)
        rank[order] = np.arange(len(q)) - np.repeat(starts, sizes)
        cols += [s, gmax - s, rank]
    cols += [(S[:, :3] > 0).sum(1) + (S[:, 3] > 0), np.repeat(sizes, sizes)]
    return np.column_stack(cols).astype(np.float32)
 
 
SCORERS = [("cname", "ratio", fuzz.ratio), ("cname", "token_set", fuzz.token_set_ratio),
           ("cname", "token_sort", fuzz.token_sort_ratio), ("cname", "partial", fuzz.partial_ratio),
           ("cname", "jaro_winkler", JaroWinkler.normalized_similarity),
           ("nospace", "ratio", fuzz.ratio), ("caddr", "token_set", fuzz.token_set_ratio),
           ("caddr", "token_sort", fuzz.token_sort_ratio), ("caddr", "ratio", fuzz.ratio),
           ("digits", "token_set", fuzz.token_set_ratio), ("akeys", "token_set", fuzz.token_set_ratio)]
FLAG_FEATURES = ["a_blank", "q_blank", "p_blank", "nonlatin", "len_diff"]
PRE_FEATURES = [f"{s}_{k}" for s in ("l1", "l2", "l3", "ak") for k in ("score", "gap", "rank")] \
    + ["n_layers", "n_cands", "is_s3"]
RICH_FEATURES = PRE_FEATURES + ["pf", "pf_gap", "pf_rank"] \
    + [f"{col}_{nm}" for col, nm, _ in SCORERS] + FLAG_FEATURES
 
 
def string_features(qt, pt, q, p):
    """String similarities + flags for pairs (q[i], p[i]); qt/pt are the prepared
    text tables aligned to qids/pids. Chunked to bound memory."""
    Q = {c: qt[c].to_numpy() for c in ("cname", "caddr", "digits", "nospace", "akeys")}
    P = {c: pt[c].to_numpy() for c in ("cname", "caddr", "digits", "nospace", "akeys")}
    qlen, plen = qt["cname"].str.len().to_numpy(), pt["cname"].str.len().to_numpy()
    q_nl, p_nl = qt["nonlatin"].to_numpy(), pt["nonlatin"].to_numpy()
    out = np.empty((len(q), len(SCORERS) + len(FLAG_FEATURES)), dtype=np.float32)
    for i in bar(range(0, len(q), FEAT_CHUNK), desc="string features", unit="chunk"):
        qs, ps = q[i:i + FEAT_CHUNK], p[i:i + FEAT_CHUNK]
        n = len(qs)
        for k, (col, _, sc) in enumerate(SCORERS):
            out[i:i + n, k] = cpdist(Q[col][qs].tolist(), P[col][ps].tolist(),
                                     scorer=sc, workers=-1)
        j = len(SCORERS)
        out[i:i + n, j] = (Q["caddr"][qs] == "") | (P["caddr"][ps] == "")
        out[i:i + n, j + 1] = Q["cname"][qs] == ""
        out[i:i + n, j + 2] = P["cname"][ps] == ""
        out[i:i + n, j + 3] = q_nl[qs] | p_nl[ps]
        out[i:i + n, j + 4] = np.abs(qlen[qs] - plen[ps])
    return out

In [7]:
# %% [CELL 5] v2 features: transliteration, legal form, skeleton (shared)
# ---------------------------------------------------------------------------
# Installs indic_transliteration if missing.
# ---------------------------------------------------------------------------
# ---- v2: transliteration, legal form, phonetic skeleton (shared) -------------
import re as _re
import unicodedata as _ud
try:
    from indic_transliteration import sanscript as _sanscript
    from indic_transliteration.sanscript import transliterate as _translit
except ImportError:
    print("installing indic_transliteration (MIT licence; needs Internet ON) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "indic_transliteration"], check=True)
    from indic_transliteration import sanscript as _sanscript
    from indic_transliteration.sanscript import transliterate as _translit
 
_BLOCK_SCRIPT = {0x0900: _sanscript.DEVANAGARI, 0x0980: _sanscript.BENGALI,
                 0x0A00: _sanscript.GURMUKHI, 0x0A80: _sanscript.GUJARATI,
                 0x0B00: _sanscript.ORIYA, 0x0B80: _sanscript.TAMIL,
                 0x0C00: _sanscript.TELUGU, 0x0C80: _sanscript.KANNADA,
                 0x0D00: _sanscript.MALAYALAM}
_LEGAL_TOKENS = {"pvt", "private", "pr", "pra", "ltd", "limited", "li", "llp", "pllc", "llc",
                 "lp", "inc", "incorporated", "corp", "corporation", "co", "company", "sas",
                 "sasu", "sarl", "eurl", "snc", "sa", "public"}
_LEGAL_PREFIX = ("prai", "pirai", "limi", "elel")    # transliterated private / limited / LLP
_FORM_KEYS = (("LLP", {"llp"}), ("PLLC", {"pllc"}), ("LLC", {"llc"}), ("LP", {"lp"}),
              ("INC", {"inc", "incorporated"}), ("CORP", {"corp", "corporation"}),
              ("SASU", {"sasu"}), ("SAS", {"sas"}), ("SARL", {"sarl"}), ("EURL", {"eurl"}),
              ("SNC", {"snc"}), ("SA", {"sa"}))
_SKEL_FOLD = (("ph", "f"), ("th", "t"), ("sh", "s"), ("kh", "k"), ("gh", "g"), ("bh", "b"),
              ("dh", "d"), ("ch", "c"), ("ck", "k"), ("c", "k"), ("q", "k"), ("x", "ks"),
              ("z", "s"), ("w", "v"), ("j", "g"), ("y", ""), ("v", "b"))   # Bengali renders B as v
 
 
def latin_tokens(name):
    """Lowercase Latin tokens of a name. Indic-script words are transliterated
    (ITRANS), accents removed, dots dropped (L.L.C -> llc), other punctuation
    splits words (Housing-Network -> housing network)."""
    s = str(name)
    s = _re.sub(r"\.(com|net|org|co|in|fr|uk|biz|info|io)\b", " ", s, flags=_re.I)
    s = s.replace(".", "")
    fast = all(ord(ch) <= 0x024F for ch in s)
    if fast:
        s = _re.sub(r"[^\w\s]", " ", s)
    else:
        # Indic vowel signs / virama are not \w, so only replace real punctuation
        # and symbols (Unicode categories P* and S*).
        s = "".join(" " if _ud.category(ch)[0] in "PS" else ch for ch in s)
    out = []
    for w in s.split():
        indic = False
        if not fast:
            blk = next((b for b in _BLOCK_SCRIPT if any(b <= ord(c) < b + 0x80 for c in w)), None)
            if blk is not None:
                indic = True
                try:
                    w = _translit(w, _BLOCK_SCRIPT[blk], _sanscript.ITRANS)
                except Exception:
                    pass
                if blk == 0x0B80:                     # Tamil: library leaves some letters
                    w = w.replace("\u0BA9", "n").replace("\u0BB1", "r")
        w = _ud.normalize("NFKD", w).encode("ascii", "ignore").decode().lower()
        if indic and blk == 0x0B80:
            # Tamil has no aspirated/voiced contrast; the library writes ச க ப ட/த
            # as jh gh bh dh — fold them to s k p t.
            for a, b in (("jh", "s"), ("gh", "k"), ("bh", "p"), ("dh", "t")):
                w = w.replace(a, b)
        w = _re.sub(r"[^a-z0-9]", "", w)
        if indic:
            w = _re.sub(r"([bcdfghjklmnpqrstvwxyz])a$", r"\1", w)    # inherent final 'a'
        if w:
            out.append(w)
    return out
 
 
def legal_form(toks):
    """Canonical legal form: PVT (private limited), PLC, LTD, LLP, LLC, INC, ...
    Spelling variants map to one code (Pvt Ltd = Private Limited = प्रा. लि.)."""
    s = set(toks)
    has_pvt = bool(s & {"pvt", "private", "pr", "pra"}) or any(t.startswith(("prai", "pirai")) for t in toks)
    has_ltd = bool(s & {"ltd", "limited", "li"}) or any(t.startswith("limi") for t in toks)
    if has_pvt:
        return "PVT"
    if "public" in s and has_ltd:
        return "PLC"
    if any(t.startswith("elel") for t in toks):
        return "LLP"
    for code, keys in _FORM_KEYS:
        if s & keys:
            return code
    if has_ltd:
        return "LTD"
    if s & {"co", "company"}:
        return "CO"
    return ""
 
 
def skeleton(toks):
    """Phonetic consonant skeleton of the non-legal tokens: folds spelling variants,
    drops vowels, collapses repeats. 'teknolojis' and 'technologies' -> 'tknlgs'."""
    words = [t for t in toks if t not in _LEGAL_TOKENS and not t.startswith(_LEGAL_PREFIX)]
    s = " ".join(words)
    for a, b in _SKEL_FOLD:
        s = s.replace(a, b)
    s = _re.sub(r"[aeiou]", "", s)
    s = _re.sub(r"(.)\1+", r"\1", s).replace("m", "n")
    return _re.sub(r"\s+", " ", s).strip()
 
 
def prepare_extra(names):
    """Per-record legal form and phonetic skeleton."""
    forms, skels = [], []
    for nm in names:
        toks = latin_tokens(nm)
        forms.append(legal_form(toks))
        skels.append(skeleton(toks))
    return pd.DataFrame({"form": forms, "skel": skels})
 
 
EXTRA_FEATURES = ["form_same", "form_diff", "form_one_missing", "name_exact",
                  "skel_ratio", "skel_tset"]
RICH_V2 = RICH_FEATURES + EXTRA_FEATURES          # v1 features first, then v2 additions
 
 
def string_features_v2(qt, pt, q, p):
    """v1 string features + legal form relation, exact name, skeleton similarity.
    qt/pt must include the v2 columns 'form' and 'skel'."""
    base = string_features(qt, pt, q, p)
    fq, fp = qt["form"].to_numpy()[q], pt["form"].to_numpy()[p]
    cq, cp = qt["cname"].to_numpy()[q], pt["cname"].to_numpy()[p]
    sq, sp = qt["skel"].to_numpy()[q].tolist(), pt["skel"].to_numpy()[p].tolist()
    both = (fq != "") & (fp != "")
    extra = np.column_stack([
        both & (fq == fp),
        both & (fq != fp),
        (fq == "") != (fp == ""),
        (cq != "") & (cq == cp),
        cpdist(sq, sp, scorer=fuzz.ratio, workers=-1),
        cpdist(sq, sp, scorer=fuzz.token_set_ratio, workers=-1),
    ]).astype(np.float32)
    return np.column_stack([base, extra]).astype(np.float32)
 
 
def ensure_extra(prep_dir, ids_path, side, raw_names_by_id):
    """Cache <side>_x.parquet (form, skel) aligned to ids; compute if missing."""
    path = prep_dir / f"{side}_x.parquet"
    if path.exists():
        return
    ids = np.load(ids_path, allow_pickle=True)
    names = raw_names_by_id.reindex(ids)
    assert names.notna().all(), f"{path}: ids missing from the source files"
    t = time.time()
    x = prepare_extra(names.tolist())
    x.insert(0, "entity_id", ids)
    x.to_parquet(path, index=False)
    log(f"{path.parent.name}/{path.name}: {len(ids):,} rows in {time.time()-t:.0f}s")
 
 
def load_prepared(prep_dir, side):
    """v1 prepared text table + v2 extra columns, row-aligned."""
    t = pd.read_parquet(prep_dir / f"{side}.parquet")
    x = pd.read_parquet(prep_dir / f"{side}_x.parquet")
    assert (t["entity_id"].values == x["entity_id"].values).all(), "extra columns misaligned"
    t["form"], t["skel"] = x["form"].values, x["skel"].values
    return t

In [8]:
# %% [CELL 6] v3: all layers, merge, translated names, fixed skeleton (shared with Step 1 v3)
# ---------------------------------------------------------------------------
# Candidate merge and v3 text columns.
# ---------------------------------------------------------------------------
# ---- v3: all blocking layers, translated names, fixed skeleton (Step 1/2 v3) ----
import os
from concurrent.futures import ProcessPoolExecutor as _PPE
import multiprocessing as _mp3
 
LAYERS = ["l1", "l2", "l3", "ak", "l5", "l6", "l7", "l8", "l8n", "l8a", "l2t"]
LAYER_SRC = {"old": ["l1", "l2", "l3", "ak"], "b": ["l5", "l6", "l7"], "c": ["l8", "l8n", "l8a", "l2t"]}
PRE_V3 = [f"{s}_{k}" for s in LAYERS for k in ("score", "gap", "rank")] + ["n_layers", "n_cands", "is_s3"]
RICH_V3 = (PRE_V3 + ["pf", "pf_gap", "pf_rank"] + [f"{col}_{nm}" for col, nm, _ in SCORERS]
           + FLAG_FEATURES + EXTRA_FEATURES + ["p_translated"])
 
 
def group_features_v3(q, S):
    """Per-query relative features for every layer score (q sorted)."""
    starts = np.flatnonzero(np.r_[True, q[1:] != q[:-1]])
    sizes = np.diff(np.r_[starts, len(q)])
    cols = []
    for j in range(S.shape[1]):
        s = S[:, j].astype(np.float32)
        gmax = np.repeat(np.maximum.reduceat(s, starts), sizes)
        order = np.lexsort((-s, q))
        rank = np.empty(len(q), np.float32)
        rank[order] = np.arange(len(q)) - np.repeat(starts, sizes)
        cols += [s, gmax - s, rank]
    cols += [(S > 0).sum(1), np.repeat(sizes, sizes)]
    return np.column_stack(cols).astype(np.float32)
 
 
def load_sources(paths):
    """paths: {"old": pairs.npz, "b": pairs_b.npz, "c": pairs_c.npz} -> arrays in memory."""
    D = {}
    for src, path in paths.items():
        z = np.load(path)
        D[src] = {k: z[k] for k in z.files}
    return D
 
 
def merge_chunk(D, q0, q1, npool):
    """Union of the candidate sources for queries q0..q1-1, sorted by (q, p);
    one column per layer score (0 when a source did not propose the pair)."""
    keys, parts = [], []
    for src, cols in LAYER_SRC.items():
        if src not in D:
            continue
        d = D[src]
        lo, hi = np.searchsorted(d["q"], [q0, q1])
        keys.append(d["q"][lo:hi].astype(np.int64) * npool + d["p"][lo:hi])
        parts.append((src, lo, hi))
    allk = np.concatenate(keys)
    uk, inv = np.unique(allk, return_inverse=True)
    out = {"q": (uk // npool).astype(np.int32), "p": (uk % npool).astype(np.int32),
           "S": np.zeros((len(uk), len(LAYERS)), dtype=np.float32)}
    has_label = all("label" in D[s] for s, _, _ in parts)
    if has_label:
        out["y"] = np.zeros(len(uk), dtype=np.int8)
    off = 0
    for src, lo, hi in parts:
        n = hi - lo
        idx = inv[off:off + n]                 # unique within a source: plain assignment
        for c in LAYER_SRC[src]:
            if c in D[src]:
                out["S"][idx, LAYERS.index(c)] = D[src][c][lo:hi]
        if has_label:
            out["y"][idx] = np.maximum(out["y"][idx], D[src]["label"][lo:hi])
        off += n
    return out
 
 
def skeleton3(toks):
    """Phonetic skeleton with the order fixed (m->n BEFORE collapsing repeats)."""
    words = [t for t in toks if t not in _LEGAL_TOKENS and not t.startswith(_LEGAL_PREFIX)]
    s = " ".join(words)
    for a, b in _SKEL_FOLD:
        s = s.replace(a, b)
    s = _re.sub(r"[aeiou]", "", s).replace("m", "n")
    s = _re.sub(r"(.)\1+", r"\1", s)
    return _re.sub(r"\s+", " ", s).strip()
 
 
def _x3_task(args):
    names, trans = args
    forms, skels = [], []
    for n, t in zip(names, trans):
        tr = latin_tokens(n)
        forms.append(legal_form(tr))           # form read from the original (translations may shorten it)
        skels.append(skeleton3(latin_tokens(t) if t else tr))
    return forms, skels
 
 
def ensure_x3(prep_dir, side, ids, names_by_id, trans):
    """Cache <side>_x3.parquet: legal form, fixed skeleton, and the cleaned name
    with translations substituted for non-Latin records."""
    path = prep_dir / f"{side}_x3.parquet"
    if path.exists():
        return
    t = time.time()
    names = names_by_id.reindex(ids)
    assert names.notna().all(), f"{path}: ids missing from the source files"
    names = names.tolist()
    tr = [trans.get(x, "") for x in ids]
    tasks = [(names[i:i + 100_000], tr[i:i + 100_000]) for i in range(0, len(names), 100_000)]
    forms, skels = [], []
    with _PPE(max_workers=os.cpu_count() or 2, mp_context=_mp3.get_context("fork")) as ex:
        for f_, s_ in ex.map(_x3_task, tasks):
            forms += f_; skels += s_
    base = pd.read_parquet(prep_dir / f"{side}.parquet", columns=["entity_id", "cname"])
    assert (base["entity_id"].values == ids).all(), "prepared texts misaligned"
    cname_t = base["cname"].to_numpy().copy()
    ti = np.array([i for i, x in enumerate(tr) if x], dtype=np.int64)
    if len(ti):
        cname_t[ti] = clean_name(pd.Series([tr[i] for i in ti])).to_numpy()
    x = pd.DataFrame({"entity_id": ids, "form": forms, "skel3": skels, "cname_t": cname_t})
    x["nospace_t"] = x["cname_t"].str.replace(" ", "", regex=False)
    x["translated"] = np.array([1 if v else 0 for v in tr], dtype=np.int8)
    x.to_parquet(path, index=False)
    log(f"{prep_dir.name}/{path.name}: {len(ids):,} rows ({int(x['translated'].sum()):,} translated) "
        f"in {time.time()-t:.0f}s")
 
 
def load_prepared_v3(prep_dir, side):
    """Prepared texts with v3 columns: translated cleaned names, fixed skeleton."""
    t = pd.read_parquet(prep_dir / f"{side}.parquet")
    x = pd.read_parquet(prep_dir / f"{side}_x3.parquet")
    assert (t["entity_id"].values == x["entity_id"].values).all(), "v3 columns misaligned"
    t["cname"], t["nospace"] = x["cname_t"].values, x["nospace_t"].values
    t["form"], t["skel"], t["translated"] = x["form"].values, x["skel3"].values, x["translated"].values
    return t
 
 
def string_features_v3(qt, pt, q, p):
    """v2 string features computed on translated names + fixed skeleton,
    plus a flag saying the candidate's name came from a translation."""
    base = string_features_v2(qt, pt, q, p)
    return np.column_stack([base, pt["translated"].to_numpy()[p].astype(np.float32)]).astype(np.float32)

In [9]:
# %% [CELL 7] Inputs
mdirs = []
for f in locate("matcher.txt"):
    cp = f.parent / "config.json"
    if (f.parent / "prefilter.txt").exists() and cp.exists():
        if json.load(open(cp)).get("rich_features") == RICH_V3:
            mdirs.append(f.parent)
assert mdirs, "stage2_models_v3 not found: attach the dataset holding the v3 models"
MDIR = mdirs[0]
cfg = json.load(open(MDIR / "config.json"))
if cfg.get("smoke"):
    print("WARNING: models from a Step 1 v3 SMOKE run — fine for testing, not for submission.")
prefilter = lgb.Booster(model_file=str(MDIR / "prefilter.txt"))
matcher = lgb.Booster(model_file=str(MDIR / "matcher.txt"))
if DEFAULT_VARIANT == "auto":
    DEFAULT_VARIANT = "base_owner" if cfg["one_owner"] else "base"
 
CKPT, PREP = WORK / "ckpt_test", WORK / "prep_test"
assert CKPT.exists(), f"{CKPT} missing: run in the notebook that holds the test blocking output"
COUNTRIES = sorted(d.name for d in CKPT.iterdir() if (d / "pairs.npz").exists())
BDIR = {f.parent.name: f.parent for f in locate("pairs_b.npz") if f.parent.parent.name == "ckpt_b_test"}
CCDIR = {f.parent.name: f.parent for f in locate("pairs_c.npz") if f.parent.parent.name == "ckpt_c_test"}
missing = [c for c in COUNTRIES if c not in BDIR or c not in CCDIR]
assert not missing, (f"Phase B/C test output missing for {missing}: attach C's Phase B TEST and "
                     f"Phase C TEST notebook outputs")
for c in COUNTRIES:
    assert (PREP / c / "q.parquet").exists() and (PREP / c / "p.parquet").exists(), \
        f"{PREP / c} missing: run Step 0 first"
tr_files = sorted(INPUT_ROOT.rglob("translations_test.parquet")) + sorted(WORK.glob("translations_test.parquet"))
assert tr_files, "translations_test.parquet not found: attach C's translation (test) output"
_t = pd.read_parquet(tr_files[0], columns=["entity_id", "translation"])
TRANS = dict(zip(_t["entity_id"], _t["translation"]))
del _t
OUTD = WORK / ("stage2_v3_smoke" if SMOKE else "stage2_v3_output")
PARTS = OUTD / "_parts"
PARTS.mkdir(parents=True, exist_ok=True)
print("=" * 70)
print(f"STEP 2 v3  {'SMOKE RUN' if SMOKE else 'FULL RUN'}   models: {MDIR}")
print(f"  keep {cfg['n_keep']} (unseen {cfg['n_keep_unseen']})   thresholds {cfg['thresholds']} "
      f"(unseen {cfg['threshold_unseen']:.2f})   one-owner {'ON' if cfg['one_owner'] else 'OFF'}   "
      f"held-out F0.5 {cfg['heldout_f05']:.4f}")
print(f"  countries {COUNTRIES}   translations {len(TRANS):,}")
print("=" * 70)

STEP 2 v3  FULL RUN   models: /kaggle/input/datasets/eshabachhav/stage2-models-v3/stage2_models_v3
  keep {'India': 80, 'US': 30} (unseen 80)   thresholds {'India': 0.89, 'US': 0.89} (unseen 0.89)   one-owner ON   held-out F0.5 0.9644
  countries ['France', 'India', 'US']   translations 867,245


In [10]:
# %% [CELL 8] v3 text columns for test (cached; the full run reuses them)
t0 = time.time()
need3 = [(c, s) for c in COUNTRIES for s in ("q", "p") if not (PREP / c / f"{s}_x3.parquet").exists()]
if need3:
    names = pd.concat([load_src(DATA_ROOT / "test" / f"test_source{i}.tsv") for i in (1, 2, 3)],
                      ignore_index=True).set_index("entity_id")["business_name"]
    for c, side in need3:
        ensure_x3(PREP / c, side, np.load(CKPT / c / f"{side}ids.npy", allow_pickle=True), names, TRANS)
    del names
    gc.collect()
timed("v3 text columns (reused by full run)", t0, scalable=None)
 

In [13]:
# %% [CELL 9] Per country: merge, pre-filter, keep top-N, score (checkpointed)
full_q = used_q = 0
for c in COUNTRIES:
    part = PARTS / f"{c}.npz"
    qids = np.load(CKPT / c / "qids.npy", allow_pickle=True)
    pids = np.load(CKPT / c / "pids.npy", allow_pickle=True)
    nq = min(SMOKE_QUERIES, len(qids)) if SMOKE else len(qids)
    full_q += len(qids); used_q += nq
    if part.exists():
        log(f"[{c}] already scored — skipping")
        continue
    n_keep = cfg["n_keep"].get(c, cfg["n_keep_unseen"])
    t_c = time.time()
    is_s3 = np.fromiter((x.startswith("S3-") for x in pids), dtype=np.int8, count=len(pids))
    D = load_sources({"old": CKPT / c / "pairs.npz", "b": BDIR[c] / "pairs_b.npz", "c": CCDIR[c] / "pairs_c.npz"})
    qt, pt = load_prepared_v3(PREP / c, "q"), load_prepared_v3(PREP / c, "p")
    oq, op, oprob = [], [], []
    n_pairs = 0
    for q0 in bar(range(0, nq, Q_CHUNK), desc=f"score {c}", unit="chunk"):
        q1 = min(nq, q0 + Q_CHUNK)
        t0 = time.time()
        m = merge_chunk(D, q0, q1, len(pids))
        n_pairs += len(m["q"])
        X = np.column_stack([group_features_v3(m["q"], m["S"]), is_s3[m["p"]]]).astype(np.float32)
        pf = np.concatenate([prefilter.predict(X[i:i + PRED_CHUNK])
                             for i in range(0, len(X), PRED_CHUNK)]).astype(np.float32)
        order = np.lexsort((-pf, m["q"]))
        qo = m["q"][order]
        st = np.flatnonzero(np.r_[True, qo[1:] != qo[:-1]])
        rank = np.arange(len(qo)) - np.repeat(st, np.diff(np.r_[st, len(qo)]))
        sel = rank < n_keep
        idx = order[sel]
        q_k, p_k, pf_k = m["q"][idx], m["p"][idx], pf[idx]
        s2 = np.flatnonzero(np.r_[True, q_k[1:] != q_k[:-1]])
        gap = np.repeat(pf_k[s2], np.diff(np.r_[s2, len(q_k)])) - pf_k
        timed("merge + pre-filter", t0)
        t0 = time.time()
        SF = string_features_v3(qt, pt, q_k, p_k)
        timed("string features", t0)
        t0 = time.time()
        Xr = np.column_stack([X[idx], pf_k, gap, rank[sel].astype(np.float32), SF]).astype(np.float32)
        # prob = np.concatenate([matcher.predict(Xr[i:i + PRED_CHUNK])
        #                        for i in range(0, len(Xr), PRED_CHUNK)]).astype(np.float32)
        prob = np.concatenate([matcher.predict(Xr[i:i + PRED_CHUNK], pred_early_stop=True,
                                               pred_early_stop_freq=10, pred_early_stop_margin=8.0)
                               for i in range(0, len(Xr), PRED_CHUNK)]).astype(np.float32)
        oq.append(q_k); op.append(p_k); oprob.append(prob)
        timed("matcher predict", t0)
        del m, X, pf, order, SF, Xr
        gc.collect()
    np.savez(part, q=np.concatenate(oq), p=np.concatenate(op), prob=np.concatenate(oprob), nq=np.array([nq]))
    log(f"[{c}] {nq:,} queries, {n_pairs:,} merged pairs ({n_pairs/max(nq,1):.0f}/entity) -> "
        f"kept top-{n_keep}, scored in {(time.time()-t_c)/60:.1f} min")
    del D, qt, pt
    gc.collect()
    mem(f"after {c}")

  [ 67.4 min] [France] already scored — skipping


score India:   0%|          | 0/17 [00:00<?, ?chunk/s]

string features:   0%|          | 0/2 [00:00<?, ?chunk/s]

string features:   0%|          | 0/2 [00:00<?, ?chunk/s]

string features:   0%|          | 0/2 [00:00<?, ?chunk/s]

...

string features:   0%|          | 0/1 [00:00<?, ?chunk/s]

string features:   0%|          | 0/1 [00:00<?, ?chunk/s]

string features:   0%|          | 0/1 [00:00<?, ?chunk/s]

string features:   0%|          | 0/1 [00:00<?, ?chunk/s]

  [148.4 min] [US] 663,106 queries, 188,350,468 merged pairs (284/entity) -> kept top-30, scored in 31.1 min
    [MEM] after US                     RAM=  6.6 GB  free= 25.8 GB


In [12]:
# for k, v in TIMES.items():
    # print(f"{k:<34}{v[0]/60:6.1f} min")

v3 text columns (reused by full run)   0.0 min
merge + pre-filter                  10.8 min
string features                      4.0 min
matcher predict                     47.8 min


In [17]:
# =============================================================================
# %% [CELL 10] Thresholds and the one-owner rule
# =============================================================================
R = {}
for c in COUNTRIES:
    z = np.load(PARTS / f"{c}.npz")
    r = dict(q=z["q"], p=z["p"], prob=z["prob"], nq=int(z["nq"][0]),
             qids=np.load(CKPT / c / "qids.npy", allow_pickle=True)[:int(z["nq"][0])],
             pids=np.load(CKPT / c / "pids.npy", allow_pickle=True))
    # one owner: keep a pair only if it is the highest-scoring claim on its S2/S3 record
    # best = pd.Series(r["prob"]).groupby(r["p"]).transform("max").to_numpy()
    # r["owner"] = r["prob"] >= best
    # exactly ONE owner per record: highest probability, exact ties broken by file order
    r["owner"] = pd.Series(-r["prob"]).groupby(r["p"]).rank(method="first").to_numpy() == 1
    R[c] = r
 
 
def matches_per_entity(r, t, allowed=None):
    m = r["prob"] >= t
    if allowed is not None:
        m &= allowed
    per = np.bincount(r["q"][m], minlength=r["nq"])
    has = np.bincount(r["q"], minlength=r["nq"]) > 0
    return float(per[has].mean()) if has.any() else 0.0
 
 
seen = cfg["thresholds"]
target = cfg["pred_matches_per_entity_all"]
TH = {"base": {}, "calib": {}}
for c in COUNTRIES:
    if c in seen:
        TH["base"][c] = TH["calib"][c] = seen[c]
        continue
    TH["base"][c] = cfg["threshold_unseen"]
    t_c = min(CALIB_GRID)
    for t_ in sorted((x for x in CALIB_GRID if x <= cfg["threshold_unseen"]), reverse=True):
        if matches_per_entity(R[c], t_) >= target:
            t_c = t_
            break
    TH["calib"][c] = t_c
VARIANTS = {"base": ("base", False), "base_owner": ("base", True),
            "calib": ("calib", False), "calib_owner": ("calib", True)}
 
 
def selected(c, variant):
    th_key, owner = VARIANTS[variant]
    m = R[c]["prob"] >= TH[th_key][c]
    return m & R[c]["owner"] if owner else m
 
 
print("\n" + "=" * 70)
print("THRESHOLDS AND LABEL-FREE SANITY CHECKS")
print("=" * 70)
print(f"  reference (held-out): matches/entity {target:.2f}, predicted-empty "
      f"{np.mean(list(cfg['pred_empty_rate'].values()))*100:.1f}%  (true singleton rate ~5.6%)")
print(f"  {'country':<8}{'variant':<13}{'threshold':>10}{'matches/entity':>16}{'empty rows':>12}"
      f"{'pairs removed by one-owner':>28}")
for c in COUNTRIES:
    for v in VARIANTS:
        m = selected(c, v)
        per = np.bincount(R[c]["q"][m], minlength=R[c]["nq"])
        has = np.bincount(R[c]["q"], minlength=R[c]["nq"]) > 0
        removed = int(((R[c]["prob"] >= TH[VARIANTS[v][0]][c]) & ~R[c]["owner"]).sum()) if VARIANTS[v][1] else 0
        print(f"  {c:<8}{v:<13}{TH[VARIANTS[v][0]][c]:>10.3f}{per[has].mean():>16.2f}"
              f"{(per == 0).mean()*100:>11.1f}%{removed:>28,}")


THRESHOLDS AND LABEL-FREE SANITY CHECKS
  reference (held-out): matches/entity 3.26, predicted-empty 5.8%  (true singleton rate ~5.6%)
  country variant       threshold  matches/entity  empty rows  pairs removed by one-owner
  France  base              0.890            3.54        4.7%                           0
  France  base_owner        0.890            3.33        5.4%                      54,105
  France  calib             0.890            3.54        4.7%                           0
  France  calib_owner       0.890            3.33        5.4%                      54,105
  India   base              0.890            3.26        5.8%                           0
  India   base_owner        0.890            3.25        5.9%                      13,500
  India   calib             0.890            3.26        5.8%                           0
  India   calib_owner       0.890            3.25        5.9%                      13,500
  US      base              0.890            3.30     

In [18]:
# =============================================================================
# %% [CELL 11] Write files
# =============================================================================
t0 = time.time()
 
 
def write_file(path, header, mask_fn):
    rows = 0
    with open(path, "w", encoding="utf-8") as f:
        f.write(header + "\n")
        for c in COUNTRIES:
            r = R[c]
            m = mask_fn(c)
            qa, pa = r["q"][m], r["p"][m]
            b = np.searchsorted(qa, np.arange(r["nq"] + 1))
            for i in range(r["nq"]):
                f.write(f"{r['qids'][i]}\t{','.join(r['pids'][pa[b[i]:b[i+1]]])}\n")
                rows += 1
    return rows
 
 
n_rows = write_file(OUTD / "candidate_pairs.tsv", "source1_entity_id\tcandidate_entity_ids",
                    lambda c: np.ones(len(R[c]["q"]), bool))
for v in VARIANTS:
    write_file(OUTD / f"matching_results_{v}.tsv", "source1_entity_id\tmatched_entity_ids",
               lambda c, v=v: selected(c, v))
shutil.copyfile(OUTD / f"matching_results_{DEFAULT_VARIANT}.tsv", OUTD / "matching_results.tsv")
timed("write files", t0)
for fn in sorted(OUTD.glob("*.tsv")):
    print(f"  written {OUTD.name}/{fn.name:<36} {fn.stat().st_size/1e6:8.1f} MB")

  written stage2_v3_output/candidate_pairs.tsv                    1381.5 MB
  written stage2_v3_output/matching_results.tsv                     95.6 MB
  written stage2_v3_output/matching_results_base.tsv                96.6 MB
  written stage2_v3_output/matching_results_base_owner.tsv          95.6 MB
  written stage2_v3_output/matching_results_calib.tsv               96.6 MB
  written stage2_v3_output/matching_results_calib_owner.tsv         95.6 MB


In [19]:
# =============================================================================
# %% [CELL 12] Checks + official validator
# =============================================================================
print("\n" + "=" * 70)
print("CHECKS")
print("=" * 70)
ok = all(len(np.unique(R[c]["q"].astype(np.int64) * len(R[c]["pids"]) + R[c]["p"])) == len(R[c]["q"])
         for c in COUNTRIES)
print(f"  {'PASS' if ok else 'FAIL'}  no duplicate candidates within a row")
own_ok = True
for c in COUNTRIES:
    m = selected(c, "base_owner")
    own_ok &= len(np.unique(R[c]["p"][m])) == int(m.sum())
print(f"  {'PASS' if own_ok else 'FAIL'}  one-owner variants: every S2/S3 record matched to at most one S1")
print(f"  rows written: {n_rows:,}")
if SMOKE:
    print("  (smoke files hold only the smoke slice: official validator skipped)")
    project(full_q / max(used_q, 1), f"full run = {full_q/max(used_q,1):.0f}x the smoke queries")
    print("  + official validator on the default file: ~5 min")
    print("\n  Smoke run passed. Set SMOKE = False and run the cell again for the full run.")
else:
    validator = DATA_ROOT.parent / "utils" / "validate_submission.py"
    log(f"official validator on matching_results_{DEFAULT_VARIANT}.tsv + candidate_pairs.tsv ...")
    r = subprocess.run([sys.executable, str(validator),
                        "--matching", str(OUTD / f"matching_results_{DEFAULT_VARIANT}.tsv"),
                        "--candidate", str(OUTD / "candidate_pairs.tsv"),
                        "--test-dir", str(DATA_ROOT / "test"), "--check-ids"],
                       capture_output=True, text=True)
    print(r.stdout[-2500:] or "(no output)", r.stderr[-800:])
    print(f"  exit code {r.returncode}: " + {0: "PASS", 1: "FAIL — read the issues above"}.get(
        r.returncode, "killed (out of memory?) — run validate_streaming.py instead"))
    print(f"\n  SUBMIT: {OUTD / 'matching_results.tsv'}  (= {DEFAULT_VARIANT})")
    print("  The other variants use the same candidates and format; they differ only in")
    print("  which pairs pass, so they are valid whenever the default passes.")
print(f"\nSTEP 2 v3 {'SMOKE' if SMOKE else 'FULL'} RUN COMPLETE in {(time.time()-T_START)/60:.1f} min")
mem("final")


CHECKS
  PASS  no duplicate candidates within a row
  PASS  one-owner variants: every S2/S3 record matched to at most one S1
  rows written: 1,732,544
  [177.2 min] official validator on matching_results_base_owner.tsv + candidate_pairs.tsv ...
ML Challenge 2026 — submission validator
  test dir: /kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset/test
  required S1 entities: 1732544
  valid S2/S3 match IDs: 9969589
  matching_results_base_owner.tsv: 1732544 rows (100650 empty, 1631894 non-empty).
  candidate_pairs.tsv: 1732544 rows (0 empty, 1732544 non-empty).

PASS — no blocking issues found. Safe to submit.
 
  exit code 0: PASS

  SUBMIT: /kaggle/working/stage2_v3_output/matching_results.tsv  (= base_owner)
  The other variants use the same candidates and format; they differ only in
  which pairs pass, so they are valid whenever the default passes.

STEP 2 v3 FULL RUN COMPLETE in 181.4 min
    [MEM] final                        RAM=  8.3 GB  fre

In [20]:
import pandas as pd
from pathlib import Path
f = next(Path("/kaggle/input").rglob("translations_test.parquet"))
t = pd.read_parquet(f)
print(f"{len(t):,} records, {t['business_name'].nunique():,} distinct names")
print(f"empty: {(t['translation'].str.strip() == '').mean()*100:.2f}%   "
      f"still Indian script: {t['translation'].str.contains('[\u0900-\u0D7F]', regex=True).mean()*100:.2f}%")
print(t.sample(15, random_state=0)[["business_name", "translation"]].to_string())

867,245 records, 375,622 distinct names
empty: 0.00%   still Indian script: 0.00%
                                               business_name                             translation
151766                    क्लासिक मैनेजमेंट प्राइवेट लिमिटेड      Classic Management Private Limited
719206                       ಸೆವೆನ್ ಎನರ್ಜಿ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್            Seven Energy Private Limited
859570                          वन फाइनेंस स्वीट्स प्रा. लि.                 One Finance Sweets Pvt.
271033                    ईस्ट कंस्ट्रक्शंस प्राइवेट लिमिटेड      East Constructions Private Limited
411201                            ಶ್ರೀ ಫುಡ್ಸ್ ಬೇಕರಿ ಲಿಮಿಟೆಡ್               Shri Foods Bakery Limited
576998                      கிரேட் எஸ்டேட் பிரைவேட் லிமிடெட்            Great Estate Private Limited
462047                          Bombay फूड्स Private Limited            Bombay Foods Private Limited
547239                        Laxmi ट्रेडिंग Private Limited           Laxmi Trading Private Limited
791038   

In [21]:
# =============================================================================
# CROSS-ENCODER — EXPORT (run in the Step 2 v3 notebook, after its FULL run)
# Writes the test pairs in v3's unsure band -> /kaggle/working/ce_band_test.parquet
# (country, s1, cand, prob). Send that file to whoever runs CE-2 (GPU).
# BAND must equal CE-1b's band (read automatically from ce_config_v2.json if attached).
# =============================================================================
import json
import numpy as np, pandas as pd
from pathlib import Path

BAND = (0.02, 0.99)          # default; replaced by CE-1b's choice if ce_config_v2.json is attached
W = Path("/kaggle/working")
_cf = [f for f in list(Path("/kaggle/input").rglob("ce_config_v2.json")) + list(W.rglob("ce_config_v2.json"))
       if not json.load(open(f)).get("smoke")]
if _cf:
    BAND = tuple(json.load(open(_cf[0]))["band"])
print("band exported:", BAND)

# Countries scored with pred_early_stop_margin=8 (probabilities only exact inside 0.018-0.982):
# their band is clamped so squashed confident scores stay outside it.
MARGIN8_COUNTRIES = ["India", "US"]
CLAMP = (0.0180, 0.9820)
def band_for(c):
    return (max(BAND[0], CLAMP[0]), min(BAND[1], CLAMP[1])) if c in MARGIN8_COUNTRIES else BAND

PARTS, CKPT = W / "stage2_v3_output" / "_parts", W / "ckpt_test"
assert PARTS.exists(), "stage2_v3_output/_parts not found: run Step 2 v3 with SMOKE = False first"
out = []
for part in sorted(PARTS.glob("*.npz")):
    c = part.stem
    z = np.load(part)
    qids = np.load(CKPT / c / "qids.npy", allow_pickle=True)
    pids = np.load(CKPT / c / "pids.npy", allow_pickle=True)
    lo_c, hi_c = band_for(c)
    b = (z["prob"] >= lo_c) & (z["prob"] < hi_c)
    out.append(pd.DataFrame({"country": c, "s1": qids[z["q"][b]], "cand": pids[z["p"][b]], "prob": z["prob"][b]}))
    print(f"{c:<7} kept pairs {len(z['q']):>12,}   band ({lo_c:.4f}, {hi_c:.4f}): {int(b.sum()):>10,} ({b.mean()*100:.1f}%)")
out = pd.concat(out, ignore_index=True)
out.to_parquet(W / "ce_band_test.parquet", index=False)
print(f"\nsaved {W / 'ce_band_test.parquet'}: {len(out):,} pairs, "
      f"{(W / 'ce_band_test.parquet').stat().st_size/1e6:.0f} MB")

band exported: (0.02, 0.99)
France  kept pairs   20,756,160   band (0.0200, 0.9900):  1,812,538 (8.7%)
India   kept pairs   64,798,880   band (0.0200, 0.9820):  1,891,240 (2.9%)
US      kept pairs   19,893,180   band (0.0200, 0.9820):  1,479,066 (7.4%)

saved /kaggle/working/ce_band_test.parquet: 5,182,844 pairs, 92 MB


In [1]:
print("hi")

hi


In [2]:
from pathlib import Path
W = Path("/kaggle/working")
for p in ["stage2_v3_output/_parts/France.npz", "stage2_v3_output/_parts/India.npz",
          "stage2_v3_output/_parts/US.npz", "stage2_v3_output/candidate_pairs.tsv",
          "ckpt_test/France/qids.npy", "ckpt_test/India/pids.npy", "ckpt_test/US/pids.npy"]:
    print(f"{'OK     ' if (W / p).exists() else 'MISSING'} {p}")

OK      stage2_v3_output/_parts/France.npz
OK      stage2_v3_output/_parts/India.npz
OK      stage2_v3_output/_parts/US.npz
OK      stage2_v3_output/candidate_pairs.tsv
OK      ckpt_test/France/qids.npy
OK      ckpt_test/India/pids.npy
OK      ckpt_test/US/pids.npy


In [7]:
# =============================================================================
# CROSS-ENCODER — APPLY v2 (run in the Step 2 v3 notebook; CPU)
# Uses CE-1b's choice (ce_config_v2.json): logistic blend or LightGBM stacker,
# on the chosen band. Writes /kaggle/working/stage2_v3ce_output/ + validator.
# INPUTS  CE-1's ce_model folder (ce_config_v2.json [+ stacker.txt]) and CE-2's
#         ce_test.parquet, attached or copied into /kaggle/working.
# =============================================================================
import json, shutil, subprocess, sys
import numpy as np, pandas as pd, lightgbm as lgb
from pathlib import Path

W, I = Path("/kaggle/working"), Path("/kaggle/input")
DATA_ROOT = Path("/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset")
V3OUT, CKPT, OUTD = W / "stage2_v3_output", W / "ckpt_test", W / "stage2_v3ce_output"

# ---- stacker features (IDENTICAL in CE-1b and apply-v2) -------------------------
STACK_FEATURES = ["z_v3", "z_ce", "ce_rank", "ce_gap", "n_band", "v3_rank", "v3_gap",
                  "ent_n_conf", "is_s3", "cand_blank"]


def _logit_(p):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def stack_features(kept, band_mask, ce, cand_blank):
    """kept: DataFrame of ALL kept pairs with columns ent, cand, prob (v3).
    band_mask: bool array over kept rows (rows re-scored by the cross-encoder).
    ce: cross-encoder probability for the band rows (same order).
    cand_blank: 1 if the candidate's address is blank, for the band rows.
    Returns the feature matrix for the band rows."""
    k = kept[["ent", "prob"]].copy()
    g = k.groupby("ent")["prob"]
    k["v3_rank"] = g.rank(ascending=False, method="first")
    k["v3_gap"] = g.transform("max") - k["prob"]
    k["ent_n_conf"] = (k["prob"] >= 0.5).groupby(k["ent"]).transform("sum")
    b = k[band_mask].copy()
    b["ce"] = ce
    gb = b.groupby("ent")["ce"]
    b["ce_rank"] = gb.rank(ascending=False, method="first")
    b["ce_gap"] = gb.transform("max") - b["ce"]
    b["n_band"] = gb.transform("size")
    b["z_v3"], b["z_ce"] = _logit_(b["prob"]), _logit_(b["ce"])
    b["is_s3"] = kept.loc[band_mask, "cand"].astype(str).str.startswith("S3-").to_numpy().astype(np.float32)
    b["cand_blank"] = np.asarray(cand_blank, dtype=np.float32)
    return b[STACK_FEATURES].to_numpy(dtype=np.float32)


STACK_PARAMS = {"objective": "binary", "num_leaves": 31, "learning_rate": 0.05, "min_data_in_leaf": 50,
                "feature_fraction": 0.9, "bagging_fraction": 0.8, "bagging_freq": 1, "seed": 42, "verbose": -1}
STACK_ROUNDS = 400


cfg_f = [f for f in list(I.rglob("ce_config_v2.json")) + list(W.rglob("ce_config_v2.json"))
         if not json.load(open(f)).get("smoke")]
ce_f = list(I.rglob("ce_test.parquet")) + list(W.glob("ce_test.parquet"))
assert cfg_f and ce_f, "attach CE-1's ce_model (with ce_config_v2.json) and CE-2's ce_test.parquet"
CE = json.load(open(cfg_f[0]))
assert CE["stack_features"] == STACK_FEATURES, "stacker feature list differs: use matching CE-1b / apply-v2 files"
# STK = lgb.Booster(model_file=str(cfg_f[0].parent / "stacker.txt")) if CE["method"] == "stacker" else None
_stk = [f for f in [cfg_f[0].parent / "stacker.txt"] + list(I.rglob("stacker.txt")) + list(W.rglob("stacker.txt"))
        if f.exists()]
assert _stk or CE["method"] != "stacker", "stacker.txt not found: attach it"
STK = lgb.Booster(model_file=str(_stk[0])) if CE["method"] == "stacker" else None
print("stacker:", _stk[0] if _stk else None, "| config:", cfg_f[0])
lo, hi = CE["band"]
# Countries scored with pred_early_stop_margin=8: probabilities exact only inside 0.018-0.982,
# so their band is clamped (must match the export cell). [] if none.
MARGIN8_COUNTRIES = ["India", "US"]
CLAMP = (0.0180, 0.9820)
print(f"held-out: v3 {CE['heldout_f05_v3']:.4f} -> {CE['method']} on band {CE['band']}: {CE['heldout_f05']:.4f}")
ces = pd.read_parquet(ce_f[0]).set_index(["s1", "cand"])["ce"]
addr = pd.concat([pd.read_csv(DATA_ROOT / "test" / f"test_source{i}.tsv", sep="\t", dtype=str,
                              keep_default_na=False, usecols=[0, 2]) for i in (2, 3)], ignore_index=True)
BLANK = dict(zip(addr.iloc[:, 0], addr.iloc[:, 1].str.strip() == ""))
del addr
OUTD.mkdir(exist_ok=True)
R, stats = {}, []
for part in sorted((V3OUT / "_parts").glob("*.npz")):
    c = part.stem
    z = np.load(part)
    nq = int(z["nq"][0])
    qids = np.load(CKPT / c / "qids.npy", allow_pickle=True)[:nq]
    pids = np.load(CKPT / c / "pids.npy", allow_pickle=True)
    q, p, prob = z["q"], z["p"], z["prob"].astype(np.float64)
    kept = pd.DataFrame({"ent": q, "cand": pids[p], "prob": prob})
    lo_c, hi_c = (max(lo, CLAMP[0]), min(hi, CLAMP[1])) if c in MARGIN8_COUNTRIES else (lo, hi)
    band = (prob >= lo_c) & (prob < hi_c)
    ce = ces.reindex(pd.MultiIndex.from_arrays([qids[q[band]], pids[p[band]]])).to_numpy()
    ok = ~np.isnan(ce)
    bi = np.flatnonzero(band)
    use = np.zeros(len(q), bool); use[bi[ok]] = True          # band rows that have a cross-encoder score
    score = prob.copy()
    if use.any():
        cb = np.array([BLANK.get(x, False) for x in kept.loc[use, "cand"]], dtype=np.float32)
        if STK is not None:
            score[use] = STK.predict(stack_features(kept, use, ce[ok], cb))
        else:
            z1, z2 = _logit_(prob[use]), _logit_(ce[ok])
            zz = np.column_stack([z1, z2, z1 * z2]) @ np.asarray(CE["blend_coef"]) + CE["blend_intercept"]
            score[use] = 1 / (1 + np.exp(-zz))
    t = CE["thresholds"].get(c, CE["threshold_unseen"])
    own = pd.Series(-score).groupby(p).rank(method="first").to_numpy() == 1   # exactly one owner
    sel = (score >= t) & own                                   # threshold + one-owner
    R[c] = dict(q=q, p=p, sel=sel, qids=qids, pids=pids, nq=nq)
    per = np.bincount(q[sel], minlength=nq)
    stats.append((c, t, int(band.sum()), int((~ok).sum()), per.mean(), (per == 0).mean()))

print(f"\n  {'country':<8}{'threshold':>10}{'band pairs':>12}{'no CE score':>12}{'matches/entity':>16}{'empty rows':>12}")
for c, t, nb, nm, mpe, er in stats:
    print(f"  {c:<8}{t:>10.2f}{nb:>12,}{nm:>12,}{mpe:>16.2f}{er*100:>11.1f}%")
if any(nm > 0 for *_, nm, _, _ in [(s[0], s[1], s[2], s[3], s[4], s[5]) for s in stats]):
    print("  NOTE: some band pairs have no cross-encoder score (band mismatch between export and CE-1b?): "
          "they keep their v3 score.")
with open(OUTD / "matching_results.tsv", "w", encoding="utf-8") as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")
    for c in sorted(R):
        r = R[c]
        qa, pa = r["q"][r["sel"]], r["p"][r["sel"]]
        bnd = np.searchsorted(qa, np.arange(r["nq"] + 1))
        for i in range(r["nq"]):
            f.write(f"{r['qids'][i]}\t{','.join(r['pids'][pa[bnd[i]:bnd[i+1]]])}\n")
shutil.copyfile(V3OUT / "candidate_pairs.tsv", OUTD / "candidate_pairs.tsv")
r = subprocess.run([sys.executable, str(DATA_ROOT.parent / "utils" / "validate_submission.py"),
                    "--matching", str(OUTD / "matching_results.tsv"), "--candidate", str(OUTD / "candidate_pairs.tsv"),
                    "--test-dir", str(DATA_ROOT / "test"), "--check-ids"], capture_output=True, text=True)
print(r.stdout[-1200:], r.stderr[-400:])
print(f"exit code {r.returncode}: {'PASS' if r.returncode == 0 else 'FAIL / killed — see above'}")
print(f"SUBMIT: {OUTD / 'matching_results.tsv'}")

stacker: /kaggle/input/datasets/eshabachhav/ce-test-stacker/stacker.txt | config: /kaggle/input/datasets/eshabachhav/ce-config-v2/ce_config_v2.json
held-out: v3 0.9642 -> stacker on band [0.02, 0.99]: 0.9751


KeyboardInterrupt: 

In [8]:
# =============================================================================
# FRANCE THRESHOLD VARIANTS — recompute ONLY France at other thresholds and swap its rows
# into an existing submission file. India/US rows stay byte-identical, so a leaderboard
# change measures France alone. Run in the notebook that produced the base file.
# ~10 min for all thresholds (France only) + validator per file.
# =============================================================================
import json, subprocess, sys
import numpy as np, pandas as pd, lightgbm as lgb
from pathlib import Path

# ---- settings -------------------------------------------------------------------
# FRANCE_THRESHOLDS = [0.60, 0.80, 0.90]               # variants to write (the base uses the config's)
FRANCE_THRESHOLDS = [0.70]
BASE_DIR = "stage2_v3ce_output"                      # v4 + CE: "stage2_v4ce_output"
PARTS_DIR = "stage2_v3_output/_parts"                # v4: "stage2_v4_output/_parts"
CFG_NAME, STK_NAME = "ce_config_v2.json", "stacker.txt"   # v4: "ce_config_v4.json", "stacker_v4.txt"
SCORE_FILES = ["ce_test.parquet"]                    # v4: ["ce_test.parquet", "ce_test_v4.parquet"]

W, I = Path("/kaggle/working"), Path("/kaggle/input")
DATA_ROOT = Path("/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset")
STACK_FEATURES = ["z_v3", "z_ce", "ce_rank", "ce_gap", "n_band", "v3_rank", "v3_gap",
                  "ent_n_conf", "is_s3", "cand_blank"]
def _logit_(p):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))
def stack_features(kept, band_mask, ce, cand_blank):
    k = kept[["ent", "prob"]].copy()
    g = k.groupby("ent")["prob"]
    k["v3_rank"] = g.rank(ascending=False, method="first")
    k["v3_gap"] = g.transform("max") - k["prob"]
    k["ent_n_conf"] = (k["prob"] >= 0.5).groupby(k["ent"]).transform("sum")
    b = k[band_mask].copy()
    b["ce"] = ce
    gb = b.groupby("ent")["ce"]
    b["ce_rank"] = gb.rank(ascending=False, method="first")
    b["ce_gap"] = gb.transform("max") - b["ce"]
    b["n_band"] = gb.transform("size")
    b["z_v3"], b["z_ce"] = _logit_(b["prob"]), _logit_(b["ce"])
    b["is_s3"] = kept.loc[band_mask, "cand"].astype(str).str.startswith("S3-").to_numpy().astype(np.float32)
    b["cand_blank"] = np.asarray(cand_blank, dtype=np.float32)
    return b[STACK_FEATURES].to_numpy(dtype=np.float32)

cfg_f = [f for f in list(I.rglob(CFG_NAME)) + list(W.rglob(CFG_NAME)) if not json.load(open(f)).get("smoke")]
assert cfg_f, f"{CFG_NAME} not found"
CE = json.load(open(cfg_f[0]))
stk = [f for f in list(I.rglob(STK_NAME)) + list(W.rglob(STK_NAME))]
STK = lgb.Booster(model_file=str(stk[0])) if CE["method"] == "stacker" else None
ce_files = [f for n in SCORE_FILES for f in list(I.rglob(n)) + list(W.glob(n))]
ces = (pd.concat([pd.read_parquet(f) for f in ce_files], ignore_index=True)
       .drop_duplicates(["s1", "cand"]).set_index(["s1", "cand"])["ce"])

# ---- France scores, exactly as in the apply cell -------------------------------------
c = "France"
z = np.load(W / PARTS_DIR / f"{c}.npz")
nq = int(z["nq"][0])
qids = np.load(W / "ckpt_test" / c / "qids.npy", allow_pickle=True)[:nq]
pids = np.load(W / "ckpt_test" / c / "pids.npy", allow_pickle=True)
q, p, prob = z["q"], z["p"], z["prob"].astype(np.float64)
kept = pd.DataFrame({"ent": q, "cand": pids[p], "prob": prob})
lo, hi = CE["band"]
band = (prob >= lo) & (prob < hi)                          # France: no clamp (never scored with margin 8)
ce = ces.reindex(pd.MultiIndex.from_arrays([qids[q[band]], pids[p[band]]])).to_numpy()
ok = ~np.isnan(ce)
use = np.zeros(len(q), bool); use[np.flatnonzero(band)[ok]] = True
score = prob.copy()
addr = pd.concat([pd.read_csv(DATA_ROOT / "test" / f"test_source{i}.tsv", sep="\t", dtype=str,
                              keep_default_na=False, usecols=[0, 2]) for i in (2, 3)], ignore_index=True)
BLANK = dict(zip(addr.iloc[:, 0], addr.iloc[:, 1].str.strip() == ""))
del addr
if use.any():
    cb = np.array([BLANK.get(x, False) for x in kept.loc[use, "cand"]], dtype=np.float32)
    if STK is not None:
        score[use] = STK.predict(stack_features(kept, use, ce[ok], cb))
    else:
        z1, z2 = _logit_(prob[use]), _logit_(ce[ok])
        zz = np.column_stack([z1, z2, z1 * z2]) @ np.asarray(CE["blend_coef"]) + CE["blend_intercept"]
        score[use] = 1 / (1 + np.exp(-zz))
# own = pd.Series(-score).groupby(p).rank(method="first").to_numpy() == 1
own = np.ones(len(score), bool)
base_t = CE["thresholds"].get(c, CE["threshold_unseen"])
print(f"France base threshold {base_t:.2f} | band pairs {int(band.sum()):,} | no CE score {int((~ok).sum()):,}")

# ---- write one file per threshold: base file with France rows replaced -----------
base = pd.read_csv(W / BASE_DIR / "matching_results.tsv", sep="\t", dtype=str, keep_default_na=False)
is_fr = base["source1_entity_id"].isin(set(qids))
print(f"base file: {len(base):,} rows, {int(is_fr.sum()):,} France rows")
print(f"  {'threshold':>9}{'matches/entity':>16}{'empty rows':>12}   file")
for t in [base_t] + [x for x in FRANCE_THRESHOLDS if abs(x - base_t) > 1e-9]:
    sel = (score >= t) & own
    qa, pa = q[sel], p[sel]
    b_ = np.searchsorted(qa, np.arange(nq + 1))
    fr = {qids[i]: ",".join(pids[pa[b_[i]:b_[i+1]]]) for i in range(nq)}
    per = np.bincount(qa, minlength=nq)
    if abs(t - base_t) < 1e-9:                     # sanity: recomputed base == file on disk
        same = (base.loc[is_fr, "matched_entity_ids"].map(lambda v: ",".join(sorted(v.split(",")))) ==
                base.loc[is_fr, "source1_entity_id"].map(lambda s: ",".join(sorted(fr[s].split(","))))).mean()
        print(f"  {t:>9.2f}{per.mean():>16.2f}{(per == 0).mean()*100:>11.1f}%   (base; recomputed rows identical: {same*100:.2f}%)")
        continue
    out = base.copy()
    out.loc[is_fr, "matched_entity_ids"] = out.loc[is_fr, "source1_entity_id"].map(fr)
    fn = W / BASE_DIR / f"matching_results_france_t{t:.2f}.tsv"
    out.to_csv(fn, sep="\t", index=False)
    r = subprocess.run([sys.executable, str(DATA_ROOT.parent / "utils" / "validate_submission.py"),
                        "--matching", str(fn), "--candidate", str(W / BASE_DIR / "candidate_pairs.tsv"),
                        "--test-dir", str(DATA_ROOT / "test")], capture_output=True, text=True)
    print(f"  {t:>9.2f}{per.mean():>16.2f}{(per == 0).mean()*100:>11.1f}%   {fn.name}  "
          f"validator {'PASS' if r.returncode == 0 else 'FAIL'}")

France base threshold 0.69 | band pairs 1,812,538 | no CE score 0
base file: 1,732,544 rows, 259,452 France rows
  threshold  matches/entity  empty rows   file
       0.69            3.54        4.7%   (base; recomputed rows identical: 86.28%)
       0.70            3.53        4.7%   matching_results_france_t0.70.tsv  validator PASS


In [ ]:
# =============================================================================
# FINAL PACKAGE — run in the notebook holding stage2_v3ce_output/ (Step 2 v3 notebook).
# Attach code_bundle.zip as a dataset. Writes /kaggle/working/<TEAM>_submission.zip
# =============================================================================
TEAM = "CUDA_Cartel"                                      # <- set your team name
FINAL_MATCHING = "/kaggle/working/stage2_v3ce_output/matching_results_france_t0.80.tsv"   # <- the file that scored best on the leaderboard
FINAL_CANDIDATES = "stage2_v3ce_output/candidate_pairs.tsv"

import shutil, zipfile, importlib.metadata as md
from pathlib import Path
from IPython.display import FileLink, display
W, I = Path("/kaggle/working"), Path("/kaggle/input")
root = W / f"{TEAM}_submission"
shutil.rmtree(root, ignore_errors=True)
(root / "output").mkdir(parents=True)
shutil.copyfile(W / FINAL_MATCHING, root / "output" / "matching_results.tsv")
shutil.copyfile(W / FINAL_CANDIDATES, root / "output" / "candidate_pairs.tsv")
bundle = next(I.rglob("code_bundle.zip"), None)
if bundle is not None:
    zipfile.ZipFile(bundle).extractall(root)
else:                                                         # Kaggle may have unpacked it
    src = next(p.parent for p in I.rglob("Documentation_template.md") if (p.parent / "code").exists())
    shutil.copytree(src / "code", root / "code"); shutil.copyfile(src / "Documentation_template.md", root / "Documentation_template.md")
pkgs = ["numpy", "pandas", "scipy", "scikit-learn", "lightgbm", "rapidfuzz", "pyarrow", "torch", "transformers",
        "sentence-transformers", "huggingface-hub", "indic-transliteration", "IndicTransToolkit", "tqdm", "psutil"]
lines = []
for p in pkgs:
    try: lines.append(f"{p}=={md.version(p)}")
    except md.PackageNotFoundError: lines.append(f"# {p}: installed at runtime by the scripts (pip)")
(root / "code" / "business_entity_resolution" / "requirements.txt").write_text("\n".join(lines) + "\n")
print(open(root / "code" / "business_entity_resolution" / "requirements.txt").read())
zpath = shutil.make_archive(str(W / f"{TEAM}_submission"), "zip", str(W), f"{TEAM}_submission")
for p in sorted(root.rglob("*")):
    if p.is_file() and p.parent.name in ("output", "business_entity_resolution", f"{TEAM}_submission"):
        print(f"  {str(p.relative_to(W)):<75} {p.stat().st_size/1e6:9.1f} MB")
print(f"\nZIP: {zpath}  {Path(zpath).stat().st_size/1e9:.2f} GB")
display(FileLink(Path(zpath).name))